# Student 2 · Traffic Predictor — B. Prepare the data like the strongest model

We copy the data pipeline of **MTGNN** (Wu et al., KDD 2020), the strongest of the reference models
at our horizons. MTGNN reuses DCRNN's data script, so the numbers stay comparable with DCRNN.

MTGNN's reported METR-LA results: **MAE 2.69 at 15 min, 3.05 at 30 min**
(RMSE 5.18 and 6.17). DCRNN reports 2.77 and 3.15.

| Choice | What MTGNN / DCRNN do | Where in their code |
|---|---|---|
| Zeros (missing) in inputs | **Kept as 0**, no imputation | `generate_training_data.py` |
| Features | speed + time of day | `add_time_in_day=True` |
| Window | 12 steps in (1 h) → 12 steps out | `x_offsets=-11..0`, `y_offsets=1..12` |
| Split | windows in time order, 70 / 10 / 20 | `round(n*0.7)`, `round(n*0.2)` |
| Normalisation | z-score speed of **train inputs**, zeros included; targets stay in mph | `util.load_dataset` |
| Loss | **masked MAE**: targets equal to 0 are ignored | `trainer.py`: `masked_mae(predict, real, 0.0)` |
| Metrics | masked MAE / RMSE / MAPE per horizon | `util.metric` |

The code lives in `metr_la.py` so the LSTM and GCN + GRU notebooks use exactly the same data and metrics.

## 1. Setup

In [1]:
import os, sys
# Find metr_la.py: run from traffic_predictor/, from the repo root, or on Colab (clone the repo)
if os.path.isfile("metr_la.py"):
    ROOT = os.getcwd()
elif os.path.isfile("traffic_predictor/metr_la.py"):
    ROOT = os.path.abspath("traffic_predictor")
else:  # Colab
    !git clone -q https://github.com/ahmadbaabod/siren.git
    !pip -q install h5py
    ROOT = os.path.abspath("siren/traffic_predictor")
sys.path.insert(0, ROOT)

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import metr_la as M

RAW = os.path.join(ROOT, "data", "raw")
M.download(RAW)
print("data in", RAW)

data in /home/user/SIREN/traffic_predictor/data/raw


## 2. Build features and windows

In [2]:
speed = M.load_speeds(RAW)                 # (time, sensor) mph, zeros = missing
data = M.build_features(speed)              # (T, N, 2): speed, time of day
starts = M.split_starts(len(data))          # last input step of each window, per split

times = speed.index
rows = []
for name, t in starts.items():
    rows.append({"split": name, "windows": len(t),
                 "first input": times[t[0] - M.SEQ_IN + 1], "last target": times[t[-1] + M.SEQ_OUT]})
pd.DataFrame(rows).set_index("split")

,windows,first input,last target
split,,,
train,23974,2012-03-01 00:00:00,2012-05-23 07:40:00
val,3425,2012-05-23 05:50:00,2012-06-04 05:05:00
test,6850,2012-06-04 03:15:00,2012-06-27 23:55:00


DCRNN's published `train.npz / val.npz / test.npz` have 23,974 / 3,425 / 6,850 samples. We get the same.
We slice the windows on the fly instead of saving those files, which are about 2.7 GB.

In [3]:
assert [len(v) for v in starts.values()] == [23974, 3425, 6850]

scaler = M.StandardScaler.fit_train_inputs(data, starts["train"])
print(f"Scaler from train inputs (zeros included): mean {scaler.mean:.3f}, std {scaler.std:.3f} mph")

ds = {k: M.WindowDataset(data, v, scaler) for k, v in starts.items()}
x, y = ds["train"][0]
print("one sample  x:", tuple(x.shape), "(12 steps, 207 sensors, [z-speed, time of day])")
print("            y:", tuple(y.shape), "(12 future steps, 207 sensors, mph)")

Scaler from train inputs (zeros included): mean 54.406, std 19.494 mph
one sample  x: (12, 207, 2) (12 steps, 207 sensors, [z-speed, time of day])
            y: (12, 207) (12 future steps, 207 sensors, mph)


### How many targets the mask removes

The masked metrics skip these readings. The test split has more missing data than train, so we must mask it
or the scores are not comparable with the papers.

In [4]:
def target_zero_share(t):
    rows = t[:, None] + np.arange(1, M.SEQ_OUT + 1)[None, :]
    return (data[rows, :, 0] == 0).mean()

for k, t in starts.items():
    print(f"{k:5s}: {target_zero_share(t):.2%} of target readings are 0 -> ignored by the masked metrics")

train: 7.13% of target readings are 0 -> ignored by the masked metrics
val  : 6.97% of target readings are 0 -> ignored by the masked metrics
test : 12.11% of target readings are 0 -> ignored by the masked metrics


## 3. Check the metrics with a naive forecast

Before training anything, score a forecast that just **repeats the last real reading** (skipping zeros).
Any model we train must beat this. It also shows why masking matters.

In [5]:
ffill = speed.replace(0, np.nan).ffill().fillna(0).values.astype(np.float32)   # last real reading

def naive_preds(t):
    return torch.from_numpy(np.repeat(ffill[t][:, None, :], M.SEQ_OUT, axis=1))   # (n, 12, N)

def targets(t):
    rows = t[:, None] + np.arange(1, M.SEQ_OUT + 1)[None, :]
    return torch.from_numpy(data[rows, :, 0])

t = starts["test"]
p, yt = naive_preds(t), targets(t)
naive = M.horizon_report(p, yt)
print("Naive 'repeat last reading' on the TEST split (masked metrics):")
display(naive)

unmasked = {h: torch.mean(torch.abs(p[:, i] - yt[:, i])).item() for h, i in M.HORIZONS.items()}
print("Same forecast scored WITHOUT the mask (MAE):", {k: round(v, 2) for k, v in unmasked.items()})
print("-> without the mask the zeros add large fake errors, so scores can't be compared with the papers.")

Naive 'repeat last reading' on the TEST split (masked metrics):


,MAE,RMSE,MAPE %
5 min,2.814,4.626,6.487
15 min,3.498,6.416,8.608
30 min,4.205,8.049,10.852


Same forecast scored WITHOUT the mask (MAE): {'5 min': 9.55, '15 min': 10.15, '30 min': 10.76}
-> without the mask the zeros add large fake errors, so scores can't be compared with the papers.


## 4. Data loaders for the next notebooks

In [6]:
loaders = {
    "train": DataLoader(ds["train"], batch_size=64, shuffle=True),
    "val":   DataLoader(ds["val"],   batch_size=64),
    "test":  DataLoader(ds["test"],  batch_size=64),
}
xb, yb = next(iter(loaders["train"]))
print("batch x:", tuple(xb.shape), " batch y:", tuple(yb.shape))

# Training step used by every model from now on (as in MTGNN's trainer):
#   pred_mph = scaler.inverse_transform(model(xb))      # model outputs z-scores
#   loss = M.masked_mae(pred_mph, yb)                    # ignore targets == 0
#   loss.backward(); clip_grad_norm_(params, 5); optimizer.step()

batch x: (64, 12, 207, 2)  batch y: (64, 12, 207)


## 5. Results table so far

| Model | 5 min MAE | 15 min MAE | 30 min MAE |
|---|---|---|---|
| DCRNN (paper) | — | 2.77 | 3.15 |
| DCRNN (official code README) | 2.18 | 2.67 | 3.08 |
| MTGNN (paper) | — | 2.69 | 3.05 |
| Naive: repeat last reading (ours) | 2.81 | 3.50 | 4.21 |
| LSTM baseline (ours) | next notebook | | |
| GCN + GRU (ours) | | | |